# Amazon Co-Purchasing Network: End-to-End GNN Pipeline

This notebook takes a real Amazon product co-purchasing dataset and runs the full graph ML pipeline:

**Raw data → Neo4j graph database → PyTorch Geometric → GNN link prediction**

Every step is shown explicitly — no pre-processed datasets, no skipped steps.

---

## About the Dataset

This dataset comes from the [Stanford Network Analysis Platform (SNAP)](https://snap.stanford.edu/data/com-Amazon.html). It captures Amazon's product co-purchasing network:

- **Nodes** are products sold on Amazon
- **Edges** mean "customers who bought product X also bought product Y"
- **Communities** represent product categories (e.g., books, electronics)

The network contains **334,863 products** and **925,872 co-purchase links**.

Our task: **link prediction** — can a GNN learn which products are likely to be co-purchased, based on the structure of the network?

---

## Section 1: Load and Explore the Data

In [ ]:
# ==============================================================
# Imports and Setup
# ==============================================================

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import random
from scipy.io import mmread
from collections import deque
from neo4j import GraphDatabase

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Torch imports (used in later sections)
import torch
torch.manual_seed(SEED)
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score

print("All imports loaded successfully.")

In [ ]:
# ==============================================================
# Load the Amazon Co-Purchasing Graph
# ==============================================================

# Load the adjacency matrix (Matrix Market format)
adj_sparse = mmread("data/com-Amazon.mtx")
G_full = nx.from_scipy_sparse_array(adj_sparse)

print(f"Nodes:              {G_full.number_of_nodes():,}")
print(f"Edges:              {G_full.number_of_edges():,}")
print(f"Density:            {nx.density(G_full):.6f}")
print(f"Connected components: {nx.number_connected_components(G_full)}")

### Original Amazon Product IDs

The dataset maps each internal node index to an original Amazon product ID.
These are opaque numeric identifiers (not human-readable product names) — the
purpose is to show that a mapping exists back to the original Amazon catalog.

In [ ]:
# ==============================================================
# Load Original Amazon Product ID Mapping
# ==============================================================

node_ids_raw = mmread("data/com-Amazon_nodeid.mtx")
# Note: this .mtx file is "array" format, so mmread returns a numpy array (not sparse)
node_ids = np.array(node_ids_raw).flatten().astype(int)

print(f"Number of product IDs: {len(node_ids):,}")
print(f"First 10 IDs:  {node_ids[:10]}")
print(f"Last 10 IDs:   {node_ids[-10:]}")

# Build a lookup: internal index -> original Amazon ID
idx_to_amazon_id = {i: int(node_ids[i]) for i in range(len(node_ids))}
print(f"\nExample: internal node 0 -> Amazon product ID {idx_to_amazon_id[0]}")

In [ ]:
# ==============================================================
# Degree Distribution
# ==============================================================

degrees = [d for _, d in G_full.degree()]

print(f"Min degree:    {min(degrees)}")
print(f"Max degree:    {max(degrees)}")
print(f"Mean degree:   {np.mean(degrees):.1f}")
print(f"Median degree: {np.median(degrees):.1f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(degrees, bins=100, edgecolor="black", alpha=0.7, color="#4A90D9")
ax.set_xlabel("Degree", fontsize=12)
ax.set_ylabel("Number of Nodes", fontsize=12)
ax.set_title("Degree Distribution (full graph)", fontsize=14, fontweight="bold")
ax.set_xlim(0, 50)  # Focus on the bulk of the distribution
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================
# Visualize a Small Neighborhood
# ==============================================================

# Pick a node with a moderate degree (not too sparse, not a mega-hub)
seed_node = None
for node, deg in G_full.degree():
    if 8 <= deg <= 15:
        seed_node = node
        break

# Extract 2-hop neighborhood
neighbors_1 = set(G_full.neighbors(seed_node))
neighbors_2 = set()
for n in neighbors_1:
    neighbors_2.update(G_full.neighbors(n))

subgraph_nodes = {seed_node} | neighbors_1 | neighbors_2
# Cap at 80 nodes for readability
if len(subgraph_nodes) > 80:
    subgraph_nodes = {seed_node} | neighbors_1 | set(list(neighbors_2)[:80 - len(neighbors_1) - 1])

sub = G_full.subgraph(subgraph_nodes)

fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.spring_layout(sub, seed=SEED, k=0.5)

# Color: seed=red, 1-hop=orange, 2-hop=lightblue
colors = []
for n in sub.nodes():
    if n == seed_node:
        colors.append("#E8734A")
    elif n in neighbors_1:
        colors.append("#F5A623")
    else:
        colors.append("#A8D8EA")

nx.draw_networkx_edges(sub, pos, ax=ax, alpha=0.3, edge_color="#888888")
nx.draw_networkx_nodes(sub, pos, ax=ax, node_color=colors, node_size=100, edgecolors="white", linewidths=0.5)
ax.set_title(f"2-Hop Neighborhood of Node {seed_node} (Amazon product {idx_to_amazon_id[seed_node]})", fontsize=13, fontweight="bold")
ax.axis("off")

from matplotlib.patches import Patch
legend = [Patch(color="#E8734A", label="Seed node"),
          Patch(color="#F5A623", label="1-hop neighbors"),
          Patch(color="#A8D8EA", label="2-hop neighbors")]
ax.legend(handles=legend, loc="upper left", fontsize=10)
plt.tight_layout()
plt.show()

print(f"Seed node degree: {G_full.degree(seed_node)}")
print(f"Subgraph: {sub.number_of_nodes()} nodes, {sub.number_of_edges()} edges")

---

## Section 2: Subsample the Graph

The full graph has 334K nodes — too large for a quick tutorial. We extract a connected subgraph of ~10,000 nodes using BFS from a fixed seed node. This preserves the local community structure (unlike random node sampling, which fragments the graph into disconnected pieces).

In [ ]:
# ==============================================================
# BFS Subsampling
# ==============================================================

TARGET_SIZE = 10000
BFS_SEED = 0  # Fixed seed node for reproducibility

def bfs_subsample(G, seed, target_size):
    """Extract a connected subgraph via BFS from a seed node."""
    visited = set()
    queue = deque([seed])
    visited.add(seed)

    while queue and len(visited) < target_size:
        node = queue.popleft()
        for neighbor in G.neighbors(node):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
                if len(visited) >= target_size:
                    break
    return visited

sampled_nodes = bfs_subsample(G_full, BFS_SEED, TARGET_SIZE)
G_sub = G_full.subgraph(sampled_nodes).copy()

# Build mapping: original node ID -> new index (0..N-1)
orig_to_new = {orig: new for new, orig in enumerate(sorted(G_sub.nodes()))}
G_sub = nx.relabel_nodes(G_sub, orig_to_new)
sampled_orig_nodes = sorted(sampled_nodes)  # Keep for community filtering

print(f"Subsampled graph:")
print(f"  Nodes: {G_sub.number_of_nodes():,}")
print(f"  Edges: {G_sub.number_of_edges():,}")
print(f"  Density: {nx.density(G_sub):.6f}")
print(f"  Connected: {nx.is_connected(G_sub)}")

print(f"\nOriginal graph:")
print(f"  Nodes: {G_full.number_of_nodes():,}")
print(f"  Edges: {G_full.number_of_edges():,}")
print(f"  Density: {nx.density(G_full):.6f}")

In [ ]:
# ==============================================================
# Filter Community Assignments to Subsampled Nodes
# ==============================================================

communities_sparse = mmread("data/com-Amazon_Communities_all.mtx").tocsr()
print(f"Full community matrix: {communities_sparse.shape[0]:,} nodes x {communities_sparse.shape[1]:,} communities")

# Slice rows to only our sampled nodes (using original indices)
comm_sub = communities_sparse[sampled_orig_nodes, :]

# Drop communities with fewer than 2 members in the subsample
col_counts = np.array((comm_sub > 0).sum(axis=0)).flatten()
keep_cols = np.where(col_counts >= 2)[0]
comm_sub = comm_sub[:, keep_cols]

# Save per-column member counts AFTER filtering (used later in t-SNE visualization)
comm_col_counts = np.array((comm_sub > 0).sum(axis=0)).flatten()

print(f"Filtered community matrix: {comm_sub.shape[0]:,} nodes x {comm_sub.shape[1]} communities")
print(f"  Communities with 2+ members in subsample: {len(keep_cols)}")

nodes_with_community = (comm_sub.sum(axis=1) > 0).sum()
print(f"  Nodes with at least one community: {int(nodes_with_community):,} / {comm_sub.shape[0]:,}")